# 07 — History & Streaming

## Objectives

- Accept that agents are **stateless** — and make memory out of message lists
- Carry context with `message_history=` (and know `all_messages` vs `new_messages`)
- Serialize a conversation for storage with `ModelMessagesTypeAdapter`
- Stream output as it's generated with `run_stream` + `stream_output`

In [ ]:
# ── SETUP — run this first in every notebook ─────────────────────────────────
import os
import sys

import logfire
from dotenv import find_dotenv, load_dotenv

# 1) Environment: walk up from notebooks/ to the repo root and load .env
load_dotenv(find_dotenv())

# 2) Observability: no Logfire account yet? This still works — data only ships
#    when a token exists (`uv run logfire auth`, or a LOGFIRE_TOKEN in .env).
logfire.configure(send_to_logfire="if-token-present")
logfire.instrument_pydantic_ai()  # v2: instrumentation is explicit — Agent(instrument=True) is gone

MODEL = os.getenv("MODEL", "openai:gpt-6.1-sol")

print("kernel :", sys.executable)  # should end in learningdantic/.venv/bin/python
print("MODEL  :", MODEL)

## Agents have no memory

Every `agent.run()` is independent: the model forgets you the millisecond the run ends.
"Memory" is you feeding old messages back in with `message_history=`. That's it —
there is no hidden state, which is exactly what makes this testable and portable.

In [ ]:
from pydantic_ai import Agent

planner = Agent(
    MODEL,
    instructions="You are a laconic travel planner. Keep answers under 25 words unless asked otherwise.",
)

first = await planner.run("I have one day in Paris. Pick me one theme for the day.")
print("turn 1:", first.output)

second = await planner.run(
    "Turn that theme into a 3-stop itinerary.",
    message_history=first.all_messages(),
)
print("turn 2:", second.output)

> ✅ **Checkpoint** — turn 2 used the theme *without you repeating it*. The conversation
> "remembered" because you carried the messages yourself.

## The carry pattern, and `all_messages` vs `new_messages`

In real code you keep one `history` variable and re-assign it after every run:

- `result.all_messages()` — the **whole** transcript (what you fed in + this run)
- `result.new_messages()` — **just this run's** messages (what a server would append to a DB)

In [ ]:
history = first.all_messages()
for ask in ["Swap the last stop for something cheaper.", "Now summarize the plan in one line."]:
    r = await planner.run(ask, message_history=history)
    history = r.all_messages()
    print("→", r.output)

print("all messages :", len(second.all_messages()), "(whole transcript)")
print("new messages :", len(second.new_messages()), "(just this run)")

## Serialize the conversation

`ModelMessagesTypeAdapter` turns a message list into JSON and back — this is how a
conversation gets stored in a file, a database, or shipped between services. In the
FastAPI phase, this exact blob is what you'll persist per user.

In [ ]:
from pydantic_ai.messages import ModelMessagesTypeAdapter

blob = ModelMessagesTypeAdapter.dump_json(history)
print("saved the conversation as", len(blob), "bytes of JSON")

restored = ModelMessagesTypeAdapter.validate_json(blob)  # what a server would load back
r = await planner.run("Which theme did I originally pick?", message_history=restored)
print(r.output)

> ✅ **Checkpoint** — the planner answered a question about turn 1 using only the
> serialized-and-restored copy of history. Persistence, proven.

## Streaming

`run_stream` gives you output as it's generated — the difference between waiting 10
seconds for a wall of text and watching it appear. `debounce_by` batches rapid chunks
so you're not printing per-token:

In [ ]:
chunks = []
async with planner.run_stream("Write a 6-line poem about debugging at 2am.") as stream:
    async for text in stream.stream_output(debounce_by=0.1):
        chunks.append(text)
        print(text, end="", flush=True)
print()

print("---")
print("stream.response:", stream.response)  # the final ModelResponse, available after the run

> ✅ **Checkpoint** — the poem printed progressively. After the block ends,
> `stream.response` holds the final `ModelResponse` — with structured output types,
> streaming still ends in a validated object.
>
> One for later: `run_stream_events` is the *event-level* firehose (every part start/delta)
> — it's what powers chat UIs in the FastAPI phase. You don't need it yet.

## 🏋️ Exercise — the Chat class

Wrap memory into a class:

```python
class Chat:
    def __init__(self, agent): ...
    async def ask(self, prompt) -> str: ...   # carries history between turns
    def save(self, path): ...                 # ModelMessagesTypeAdapter → file
    @classmethod
    def load(cls, agent, path) -> Chat: ...
```

Then have a 3+ turn conversation with one agent, `save` it, load it into a **fresh** `Chat`,
and prove the new object remembers. (This class is 90% of "chatbot backend".)

Your take: [`../solutions/07_chat_class.ipynb`](../solutions/07_chat_class.ipynb).

## Next

→ **[08_offline_testing.ipynb](08_offline_testing.ipynb)** — everything so far cost tokens
to verify. Time to make that free.